# 🌾 Atelier IA Agricole — 01. LLM avec Ollama

Un **LLM** (*Large Language Model*) est un modèle entraîné sur d'énormes quantités de texte.
Il sait **répondre à des questions**, **résumer**, **traduire**, **rédiger**…

Dans ce notebook, on l'utilise comme **assistant agronome** :
- charger un LLM avec Ollama ;
- lui poser des questions agricoles ;
- découvrir l'**ingénierie de prompt** (l'art de bien formuler ses demandes) ;
- comprendre le paramètre **température**.

> ⚙️ Exécutez d'abord la cellule de configuration, puis suivez l'ordre des cellules.


In [1]:
# === Configuration de l'environnement (exécuter en premier) ===
import os, sys, subprocess

# Sommes-nous dans Google Colab ?
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# MODE_DEMO = True  -> utilise de tout petits modèles / jeux de données réduits
# (utile pour tester rapidement, ou hors GPU). Mettez False pour l'atelier complet.
# La variable d'environnement ATELIER_DEMO permet de forcer le mode pour les tests.
MODE_DEMO = os.environ.get("ATELIER_DEMO", "0") == "1"

def pip_install(*paquets):
    """Installe des paquets pip de façon silencieuse (Colab ou local)."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *paquets]
    print("Installation :", " ".join(paquets), "...")
    subprocess.run(cmd, check=False)

print(f"IS_COLAB = {IS_COLAB}")
print(f"MODE_DEMO = {MODE_DEMO}")
print("Python :", sys.version.split()[0])


IS_COLAB = False
MODE_DEMO = False
Python : 3.12.3


## 1. Préparer Ollama

On réutilise les fonctions du notebook 00 : démarrer le serveur, puis discuter avec un modèle.
(La cellule réinstalle/relance ce qu'il faut, au cas où vous repartez de zéro.)


In [2]:
import os, subprocess, shutil, time, json, urllib.request

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

def chemin_ollama():
    candidats = [
        shutil.which("ollama"),
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
        "/root/.local/bin/ollama",
    ]
    for candidat in candidats:
        if candidat and os.path.exists(candidat):
            return candidat
    return None

if chemin_ollama() is None and IS_COLAB:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=False)
    time.sleep(2)

OLLAMA_CMD = chemin_ollama()

def serveur_actif():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not serveur_actif():
    if OLLAMA_CMD:
        subprocess.Popen([OLLAMA_CMD, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            if serveur_actif():
                break
            time.sleep(1)
    else:
        print("⚠️ Ollama n'est pas disponible. En local, installez-le depuis https://ollama.com/download")

print("Serveur Ollama actif :", serveur_actif())


Serveur Ollama actif : True


## 2. Choisir et charger un modèle

On définit le modèle dans **une seule variable** : `MODELE`.
- En **mode démo** (test rapide) : un petit modèle.
- En **mode complet** (atelier) : `llama3.2` (~2 Go), plus performant.

👉 Pour changer de modèle, il suffira de modifier cette variable (voir les exercices).


In [3]:
MODELE = "qwen2.5:0.5b" #if MODE_DEMO else "llama3.2"

print(f"Téléchargement du modèle : {MODELE} ...")
subprocess.run(["ollama", "pull", MODELE], check=False)
print("✅ Modèle prêt.")


Téléchargement du modèle : qwen2.5:0.5b ...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ 

✅ Modèle prêt.


pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling c5396e06af29: 100% ▕██████████████████▏ 397 MB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 005f95c74751: 100% ▕██████████████████▏  490 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
def chat(prompt, modele=None, temperature=0.7, systeme=None):
    """Interroge le LLM. `systeme` = consigne de rôle ; `temperature` = créativité (0=strict)."""
    modele = modele or MODELE
    corps = {
        "model": modele,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    }
    if systeme:
        corps["system"] = systeme
    donnees = json.dumps(corps).encode()
    req = urllib.request.Request("http://localhost:11434/api/generate", data=donnees,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.loads(r.read())["response"]

# Premier test
print(chat("En une phrase, qu'est-ce que la rotation des cultures ?"))


La rotation des cultures est un processus de transition et d'adaptation qui vise à fournir aux populations vivant dans les différentes régions géographiques l'éducation et l'apprentissage nécessaire pour se sentir connectées avec leur communauté nationale. Cela implique la mise en place d'une structure de communication éthérique, des systèmes éducatifs adaptés aux traditions locales, et une compréhension mutuelle entre les populations vivant dans différentes régions.

Ces rotations des cultures contribuent à un respect mutuel pour l'identité nationale, favorisent la cohésion sociale et culturelle en améliorant ainsi le bien-être général de l'ensemble du pays. Leur importance vaut particulièrement aujourd'hui face aux défis du changement climatique et des obstacles aux droits humains, car les cultures sont souvent les seules sources d'identité et de connexion au sein de leur communauté.

En résumé, la rotation des cultures est un processus qui tente de renforcer l'unité nationale en amé

## 3. Un assistant agronome

Le paramètre **`system`** donne un **rôle** au modèle. Comparez la différence de ton :


In [5]:
consigne = ("Tu es un ingénieur agronome francophone. "
            "Tu réponds de façon claire, pratique et concise, pour de petits agriculteurs.")

question = "Mes feuilles de tomate ont des taches jaunes. Quelles peuvent être les causes ?"
print(chat(question, systeme=consigne))


Voici quelques raisons possibles pour l'apparition de taches jaunes sur vos feuilles de tomates :

1. **Déséchement ou Excessivezage de la Tomatine**: Les tomates sont souvent sécrétantes de plusieurs minéraux et minéraux. Si elles ne peuvent plus se sécruber, cela peut entraîner un malaise de l'aspérité.

2. **Changement du pH**: La chaleur ou le sel peuvent modifier rapidement le pH des sols envers les tomates. Cela pourrait expliquer la tache jaune.

3. **Infection par micro-organismes**: Ces organismes, comme la bactérie ornementale (bacterium), peuvent avoir un impact soudain sur l'aspérité de votre tomate.

4. **Soyez prudent avec le sel et l'eau**: L'apport excessif d'eau ou de sel peut potentiellement aggraver les problèmes de santé de l'aspérité. Assurez-vous que vous utilisez toujours un sol propre.

5. **Mieux connaître vos tomates et votre environnement**: Si ces taches jaunes se produisent chez vous, il serait possible qu'il y ait des facteurs environnementaux spécifiques 

## 4. L'ingénierie de prompt (prompt engineering)

La **qualité de la réponse dépend de la façon de poser la question**.
Trois leviers simples :

1. **Donner un rôle** (« Tu es un agronome… »).
2. **Donner du contexte** (région, culture, saison…).
3. **Imposer un format** (liste, tableau, nombre de mots…).

Comparez un prompt **vague** et un prompt **précis** :


In [6]:
prompt_vague = "Parle-moi du maïs."

prompt_precis = (
    "Tu es un conseiller agricole. Pour un agriculteur en Afrique de l'Ouest qui débute, "
    "donne 3 conseils pratiques pour réussir une culture de maïs en saison des pluies. "
    "Réponds sous forme de liste à puces, une phrase par conseil."
)

print("=== PROMPT VAGUE ===")
print(chat(prompt_vague))
print("\n=== PROMPT PRÉCIS ===")
print(chat(prompt_precis))


=== PROMPT VAGUE ===
Le maïs est un légume important et souvent apprécié dans la cuisine, mais aussi en tant que source d'agrumes. Voici quelques points essentiels à connaître :

1. Type de plantes : Le maïs est une plante maroquée qui se caractérise par des graines en forme de paon.

2. Structure : Les graines sont en forme de paon et ont un coude droit (le "sourisette") et un coude gauche (le "cône" ou le "bouillon").

3. Qualités du maïs :
   - Sa texture épaisse et riche en acide.
   - Ses propriétés anti-inflamatrices et antioxydantes.
   - Une grande quantité d'acides gourmands.
   - Ses propriétés anti-carcinogènes.

4. Utilisations :
   - Jus de maïs : un jus frais du maïs est utilisé pour l'onction, la peau et les baignades.
   - Maïs en poudre : une variété de maïs (maisons) est souvent utilisée dans le vin.
   - Produits aliments : des crèmes noirs, des fromages, des salades.

5. Variétés :
   - Maïs de mélèze : un type d'agneau qui se combine avec du maïs pour équilibrer la

## 5. La température

La **température** contrôle la créativité :
- `0.0` → réponses **déterministes**, factuelles, répétitives.
- `1.0` → réponses **variées**, créatives (mais parfois moins fiables).

Pour des **conseils techniques**, une température **basse** est préférable.


In [7]:
q = "Propose un nom de marque pour un miel bio de lavande."
for t in [0.0, 1.0]:
    print(f"--- température = {t} ---")
    print(chat(q, temperature=t))
    print()


--- température = 0.0 ---
Pour une marque de miel bio de lavande, on pourrait proposer "Lavandiai Bio". C'est un nom qui reflète l'identité naturelle et originale du miel, tout en montrant que le produit est conçu pour être biologique. L'expression "Bio" indique également qu'il s'agit d'un miel bio, ce qui signifie que la production du miel n'a pas été affectée par l'utilisation de pesticides ou d'autres substances chimiques. Cela permet à l'entreprise de respecter les principes environnementaux et de se concentrer sur le produit principal, sans compromettre sa qualité originale.

--- température = 1.0 ---
Un nom de marque approprié pour une variété de miel biologique de lavande serait :

"Vermicule Lavandière"

Cela vaut autant d'important, et c'est en raison de la qualité et de l'environnement friendly que vous trouverez le mélange. Cette combinaison donne un lien direct avec les mèches de lavande et permet non seulement de faire preuve du bien-être et de l'environnement au travail m

---
# 🏋️ Exercices

Essayez par vous-même ! Les **solutions** se trouvent juste après chaque exercice
(repliez-les pour d'abord chercher seul).


### 🏋️ Exercice 1 — Changer de modèle

Téléchargez un autre petit modèle, par exemple **`gemma2:2b`** (ou `qwen2.5:0.5b` en démo),
puis posez-lui la question : *« Qu'est-ce que le paillage (mulching) ? »*
en utilisant l'argument `modele=...` de la fonction `chat`.


In [8]:
# 👉 Votre code ici


### ✅ Solution 1


In [9]:
autre_modele = "qwen2.5:0.5b" if MODE_DEMO else "gemma2:2b"
subprocess.run(["ollama", "pull", autre_modele], check=False)
print(chat("Qu'est-ce que le paillage (mulching) ?", modele=autre_modele))


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏  54 KB/1.6 GB                  pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏ 673 KB/1.6 GB                  pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏ 1.0 MB/1.6 GB                  pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏ 1.7 MB/1.6 GB                  pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏ 2.3 MB/1.6 GB                  pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏ 2.6 MB/1.6 GB                  pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏ 3.3 MB/1.6 GB                  pulling manifest 
pulling 7462734796d6:   0% ▕                  ▏ 3.9 MB/1.6 GB                  pulling manifest 
pulling 74627347

TimeoutError: timed out

### 🏋️ Exercice 2 — Améliorer un prompt

Le prompt ci-dessous est trop vague. **Réécrivez-le** en ajoutant : un rôle, un contexte
(culture = vigne, problème = mildiou) et un format (liste de 3 actions).


In [ ]:
prompt_a_ameliorer = "Comment traiter une maladie ?"
# 👉 Réécrivez ce prompt puis appelez chat(...)


### ✅ Solution 2


In [ ]:
prompt_ameliore = (
    "Tu es un conseiller viticole. Un viticulteur observe du mildiou sur sa vigne. "
    "Donne 3 actions concrètes pour limiter la propagation, sous forme de liste numérotée."
)
print(chat(prompt_ameliore))


### 🏋️ Exercice 3 — Effet de la température

Demandez au modèle d'**inventer un slogan** pour une coopérative agricole,
en testant les températures `0.2` puis `0.9`. Observez la différence de créativité.


In [ ]:
# 👉 Votre code ici


### ✅ Solution 3


In [ ]:
for t in [0.2, 0.9]:
    print(f"--- température = {t} ---")
    print(chat("Invente un slogan court pour une coopérative agricole.", temperature=t))
    print()


## ✅ Récapitulatif

- Un LLM se charge en **une variable** (`MODELE`) avec Ollama.
- La fonction `chat(...)` accepte un **rôle** (`system`) et une **température**.
- Un **bon prompt** = rôle + contexte + format.

**➡️ Notebook suivant : `02_SLM_petits_modeles.ipynb`** — les petits modèles pour le terrain.
